In [1]:
from IPython.display import display, HTML 
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from urllib.request import urlopen
import math
import time
import requests
import pandas as pd

kospi, kosdak, divident = 'gsum', 'ksd_gsum', 'div'

C:\Users\Admin\anaconda3\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (4.0.0)/charset_normalizer (2.0.4) doesn't match a supported version!
  warnings.warn(


In [3]:
def load_csv_file(str):
    date = time.strftime("%y%m%d", time.localtime())
    item_list = pd.read_csv(f'data/naver/fin_{str}_{date}.csv', encoding='cp949') 
    return item_list

## 네이버 금융/주식/시가총액 화면

In [4]:
def call_nav_fin_gsum(string):

    driver = webdriver.Chrome()
    if(string == 'gsum'):
        driver.get('https://finance.naver.com/sise/sise_market_sum.naver?sosok=0')
    elif(string == 'ksd_gsum'):
        driver.get('https://finance.naver.com/sise/sise_market_sum.naver?sosok=1')
    time.sleep(0.5) # 초기화면 접근

    no_tgl_bxs = len(driver.find_elements(By.CSS_SELECTOR, 'td > input[type="checkbox"]'))
    toggle_boxes = [f'option{i}' for i in range(1, no_tgl_bxs+1)] # 체크박스 이름 준비

    pageN = 10
    max_tgl = 6
    item_list = pd.DataFrame([])
    for page in range(1,pageN+1):
        page_item_list = []
        for sheet in range(0, math.ceil(no_tgl_bxs/max_tgl)):
            dflt_toggle_boxes = driver.find_elements(By.CSS_SELECTOR, 'td.choice > input[type="checkbox"]')
            [elem.send_keys(Keys.SPACE) for elem in dflt_toggle_boxes]
            time.sleep(0.5) # 디폴트 체크박스 toggle

            sheet_toggle_boxes = toggle_boxes[sheet*max_tgl:min((sheet+1)*max_tgl, no_tgl_bxs+1)]
            new_toggle_boxes = [select_box for select_box in sheet_toggle_boxes]
            [driver.find_element(By.ID, box_id).send_keys(Keys.SPACE) for box_id in new_toggle_boxes]
            time.sleep(0.5) # 시트별 새로운 체크박스 toggle
            driver.find_element(By.CSS_SELECTOR, 'div.item_btn > a').click()
            time.sleep(0.5) # 토글 체크후 submit버튼 클릭

            soup = BeautifulSoup(driver.page_source, 'html.parser')
            table_head = [e.text.strip() for e in soup.select('thead > tr >th')][:-1]
            table_head.insert(3, '전일비변동')
            items = soup.select('tbody > tr') # 테이블 헤드 수집
            sheet_item_list = []
            for idx, item in enumerate(items):
                if(item.select_one('td.no')):
                    no = item.select_one('td.no').text
                    title = item.select_one('a.tltle').text
                    price_vals = [e.text.strip() for e in item.select('td.number')]
                    if(price_vals[1].strip()=='보합0'):
                        price_vals[1]=['보합',0]
                    else:
                        price_vals[1] = [price_vals[1].split('\n')[0], price_vals[1].split('\n')[1].strip()]

                    price_vals.insert(0, no)
                    price_vals.insert(1, title)
                    temp = price_vals[3][1]
                    price_vals.insert(3,price_vals[3][0])
                    price_vals[4] = temp

                    sheet_item_list.append(price_vals) # 테이블 내용 수집 
            sheet_item_df = pd.DataFrame(sheet_item_list, columns=table_head)

            if sheet == 0:
                page_item_list = sheet_item_df
            elif sheet < round(no_tgl_bxs/max_tgl):
                page_item_list = pd.concat([page_item_list, sheet_item_df[table_head[7:13]]], axis=1)
            else: 
                page_item_list = pd.concat([page_item_list, sheet_item_df[table_head[7:9]]], axis=1)
                # 페이지 내 시트 축적
        item_list = pd.concat([item_list, page_item_list], axis=0) # 페이지 내용 축적

        page_div = driver.find_element(By.CSS_SELECTOR, 'table.Nnavi > tbody > tr')
        if(page<pageN): # 지정 마지막 페이지 초과(pageN+1) 이동 금지
            page_div.find_element(By.LINK_TEXT, str(page+1)).click()
            time.sleep(1.5) # next page로
    print(f'~~ 총 {page}페이지 {sheet}시트 데이터를 수집 하였습니다~~')

    item_list.to_csv(f'data/naver/fin_{string}_{time.strftime("%y%m%d", time.localtime())}.csv', index=False, encoding='cp949')
    item_list.sample(10)

## 네이버 배당 화면 접근

In [5]:
def call_nav_fin_divident(string):
    driver = webdriver.Chrome()
    driver.get('https://finance.naver.com/sise/dividend_list.naver')
    time.sleep(0.5)

    pageN = 10
    item_list = []
    for page in range(1,pageN+1):
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table_head = [e.text.strip() for e in soup.select('thead > tr > th')]
        table_head[9] = '과거3년배당금_Max'
        


        items = soup.select('table.type_1.tb_ty> tbody > tr')
        for idx, item in enumerate(items):
            if(item.text.strip()):
                price_vals = [e.text.strip().replace(',','').replace('-','0') for e in item.select('td')]
                price_vals.insert(9, max(price_vals[9:12]))

                item_list.append({table_head[i]:e for i, e in enumerate(price_vals)})

        # next page로
        page_div = driver.find_element(By.CSS_SELECTOR, 'table.Nnavi > tbody > tr')
        if(page<pageN): # 지정 마지막 페이지 초과(pageN+1) 이동 금지
            page_div.find_element(By.LINK_TEXT, str(page+1)).click()
            time.sleep(1.5)
            
    print(f'~~ 총 {page}페이지 데이터를 수집 하였습니다~~')
    df = pd.DataFrame(item_list)
    df.to_csv(f'data/naver/fin_{string}_{time.strftime("%y%m%d", time.localtime())}.csv', index=False, encoding='cp949')

In [6]:
call_nav_fin_gsum(kospi)
call_nav_fin_gsum(kosdak)
call_nav_fin_divident(divident)

kospi_item_list = load_csv_file(kospi)
kosdak_item_list = load_csv_file(kosdak)
divident_item_list = load_csv_file(divident)

kosdak_item_list.tail(3)

selected_divident_items = divident_item_list[(divident_item_list['수익률(%)']>=5.0) &
                                             (divident_item_list['ROE(%)']>=5.0) &
                                             (divident_item_list['PER(배)']<=10.0) &
                                             (divident_item_list['PBR(배)']<=1.0) &
                                             (divident_item_list['배당성향(%)']>=25.0)]
display(selected_divident_items)
p_kospi = kospi_item_list[kospi_item_list['전일비변동']=='하락'].sort_values('등락률', ascending=False)
p_kosdak = kosdak_item_list[kosdak_item_list['전일비변동']=='하락'].sort_values('등락률', ascending=False)
proposed_kospi = pd.merge(selected_divident_items, p_kospi, how='inner', on=['종목명'])
proposed_kosdak = pd.merge(selected_divident_items, p_kosdak, how='inner', on=['종목명'])

display(proposed_kospi.T)
display(proposed_kosdak.T)


~~ 총 10페이지 4시트 데이터를 수집 하였습니다~~
~~ 총 10페이지 4시트 데이터를 수집 하였습니다~~
~~ 총 10페이지 데이터를 수집 하였습니다~~


,종목명,현재가,기준월,배당금,수익률(%),배당성향(%),ROE(%),PER(배),PBR(배),과거3년배당금_Max,1년전,2년전,3년전
1,레드캡투어,11710,25.03,2150,18.36,177.53,9.64,7.26,0.67,450,450,450,400
13,HB인베스트먼트,2130,24.12,200,9.39,89.81,8.28,6.99,0.49,0,0,0,0
14,정다운,2705,24.12,250,9.24,68.61,9.33,7.31,0.65,300,300,100,100
21,노바텍,17750,24.12,1409,7.94,80.44,10.80,8.95,0.87,500,500,200,143
23,영보화학,4420,24.12,350,7.92,30.26,13.01,3.16,0.38,50,50,50,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,YW,3790,24.12,200,5.28,37.63,5.31,9.62,0.36,200,200,150,100
155,BNK금융지주,12440,25.02,650,5.22,28.46,6.96,4.56,0.31,625,510,625,560
162,대상우,17030,25.03,860,5.05,32.18,6.94,7.26,0.49,810,810,810,810
163,미원화학,79300,24.12,4000,5.04,29.79,17.55,6.44,0.98,3500,3500,2500,2500


,0,1,2,3,4,5,6,7,8,9,10
종목명,현대차우,현대차2우B,기아,LX인터내셔널,SK텔레콤,이노션,기업은행,우리금융지주,한국앤컴퍼니,BNK금융지주,코리안리
현재가_x,155800,158600,97800,31150,56100,20050,18230,22375,18850,12440,10250
기준월,25.02,25.02,25.03,25.02,25.02,25.03,25.03,25.02,25.03,25.02,25.04
배당금,12050,12100,6500,2000,3540,1175,1065,1200,1000,650,515
수익률(%),7.73,7.63,6.65,6.42,6.31,5.86,5.84,5.36,5.3,5.22,5.02
배당성향(%),25.13,25.13,26.18,40.94,60.28,46.88,32.11,28.87,27.01,28.46,28.74
ROE(%),12.43,12.43,19.09,7.12,10.83,10.48,8.06,9.38,8.28,6.96,9.44
PER(배),4.6,4.6,4.12,5.97,9.5,7.74,4.32,3.71,4.58,4.56,4.89
PBR(배),0.51,0.51,0.71,0.37,1.0,0.77,0.34,0.33,0.36,0.31,0.41
과거3년배당금_Max,7050,7100,5600,3000,3540,900,984,900,700,625,458


,0,1
종목명,레드캡투어,에스에이엠티
현재가_x,11710,2935
기준월,25.03,24.12
배당금,2150,200
수익률(%),18.36,6.81
배당성향(%),177.53,35.89
ROE(%),9.64,13.46
PER(배),7.26,4.79
PBR(배),0.67,0.59
과거3년배당금_Max,450,230
